# 02. 5단계 협업 protocol 구현

목표: P1 탐색, P2 분할, P3 실행, P4 검토, P5 제출과 전원 승인 gate를 작은 상태 기계로 구현합니다. 실제 LLM이나 network는 호출하지 않습니다.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class Phase(Enum):
    EXPLORE = 'P1'
    DIVIDE = 'P2'
    EXECUTE = 'P3'
    REVIEW = 'P4'
    SUBMIT = 'P5'
    DONE = 'DONE'

@dataclass
class Protocol:
    agents: tuple[str, ...]
    phase: Phase = Phase.EXPLORE
    approvals: set[str] = field(default_factory=set)
    worklog: list[dict] = field(default_factory=list)

    def approve(self, agent: str):
        if agent not in self.agents:
            raise ValueError(f'unknown agent: {agent}')
        self.approvals.add(agent)

    def broadcast_discovery(self, sender: str, evidence: str):
        if self.phase != Phase.EXECUTE:
            raise RuntimeError('발견 공유는 P3에서 수행합니다.')
        self.worklog.append({'sender': sender, 'evidence': evidence})

    def advance(self):
        gated = {Phase.DIVIDE, Phase.REVIEW, Phase.SUBMIT}
        if self.phase in gated and self.approvals != set(self.agents):
            missing = set(self.agents) - self.approvals
            raise RuntimeError(f'승인 부족: {sorted(missing)}')
        order = [Phase.EXPLORE, Phase.DIVIDE, Phase.EXECUTE, Phase.REVIEW, Phase.SUBMIT, Phase.DONE]
        self.phase = order[order.index(self.phase) + 1]
        self.approvals.clear()
        return self.phase

In [ ]:
team = ('agent-1', 'agent-2', 'agent-3', 'agent-4')
p = Protocol(team)
assert p.advance() == Phase.DIVIDE

try:
    p.advance()
except RuntimeError as exc:
    print('gate 동작:', exc)

for agent in team:
    p.approve(agent)
assert p.advance() == Phase.EXECUTE
p.broadcast_discovery('agent-4', 'MINIO_AUDIT_WEBHOOK_ENABLE 발견')
p.broadcast_discovery('agent-1', 'audit webhook을 공유 server에 연결')
print(p.worklog)
assert p.advance() == Phase.REVIEW

In [ ]:
for phase in (Phase.REVIEW, Phase.SUBMIT):
    assert p.phase == phase
    for agent in team:
        p.approve(agent)
    p.advance()
assert p.phase == Phase.DONE
print('모든 gate를 통과했습니다:', p.phase.value)

## 확장 과제

승인만 세는 현재 구현에 evidence provenance, timeout, duplicate message id와 P4에서 P3로 돌아가는 재작업 전이를 추가해 보세요. 실제 분산 시스템에서는 agent crash와 늦게 도착한 승인을 처리하기 위한 idempotency도 필요합니다.